# Minute bars: Alpaca to Tiger Data

Loads five years of 1-minute bars for AAPL and META into the Tiger Data table `stock_minute_bars`.

META traded as `FB` until 2022-06-09. Bars before that rename are requested as `FB` and stored as `META`, so the series covers the full window. AAPL is unchanged.

Credentials come from the repo-root `.env` (see `.env.example`). Nothing in this notebook prints secret values.

Alpaca's free plan serves the `iex` feed. Set `ALPACA_DATA_FEED=sip` if the account includes SIP.

In [ ]:
import os
import time
from datetime import datetime, timedelta, timezone
from pathlib import Path

import psycopg
import requests
from dotenv import load_dotenv
from psycopg.rows import dict_row

repo_root = Path.cwd().resolve()
for candidate in [repo_root, *repo_root.parents]:
    if (candidate / ".git").exists():
        repo_root = candidate
        break
load_dotenv(repo_root / ".env")

REQUIRED = [
    "ALPACA_API_KEY",
    "ALPACA_API_SECRET",
    "TIGER_DB_PGHOST",
    "TIGER_DB_PGPORT",
    "TIGER_DB_PGDATABASE",
    "TIGER_DB_PGUSER",
    "TIGER_DB_PGPASSWORD",
]
missing = [name for name in REQUIRED if not os.environ.get(name)]
if missing:
    raise RuntimeError(
        "Missing environment variables: "
        + ", ".join(missing)
        + ". Copy .env.example to .env at the repo root and fill them in."
    )

SYMBOLS = ("AAPL", "META")
META_RENAME = datetime(2022, 6, 9, tzinfo=timezone.utc)
ALPACA_BARS_URL = "https://data.alpaca.markets/v2/stocks/bars"
DATA_FEED = os.environ.get("ALPACA_DATA_FEED", "iex")
WINDOW = timedelta(days=31)
END = datetime.now(timezone.utc).replace(second=0, microsecond=0)
START = END - timedelta(days=365 * 5)

print(f"Window {START.isoformat()} -> {END.isoformat()} feed={DATA_FEED}")

In [ ]:
conn = psycopg.connect(
    host=os.environ["TIGER_DB_PGHOST"],
    port=os.environ["TIGER_DB_PGPORT"],
    dbname=os.environ["TIGER_DB_PGDATABASE"],
    user=os.environ["TIGER_DB_PGUSER"],
    password=os.environ["TIGER_DB_PGPASSWORD"],
    sslmode=os.environ.get("TIGER_DB_PGSSLMODE", "require"),
    autocommit=True,
)

with conn.cursor() as cur:
    cur.execute("SELECT extname FROM pg_extension WHERE extname = 'timescaledb'")
    if cur.fetchone() is None:
        cur.execute("CREATE EXTENSION IF NOT EXISTS timescaledb")

    cur.execute(
        """
        CREATE TABLE IF NOT EXISTS stock_minute_bars (
            symbol TEXT NOT NULL,
            ts TIMESTAMPTZ NOT NULL,
            open DOUBLE PRECISION NOT NULL,
            high DOUBLE PRECISION NOT NULL,
            low DOUBLE PRECISION NOT NULL,
            close DOUBLE PRECISION NOT NULL,
            volume BIGINT NOT NULL,
            trade_count BIGINT,
            vwap DOUBLE PRECISION,
            PRIMARY KEY (symbol, ts)
        )
        """
    )
    cur.execute(
        """
        SELECT create_hypertable(
            'stock_minute_bars',
            'ts',
            if_not_exists => TRUE,
            migrate_data => TRUE
        )
        """
    )

print("stock_minute_bars is ready")

In [ ]:
session = requests.Session()
session.headers.update(
    {
        "APCA-API-KEY-ID": os.environ["ALPACA_API_KEY"],
        "APCA-API-SECRET-KEY": os.environ["ALPACA_API_SECRET"],
    }
)


def fetch_bars(symbols, start, end):
    """Page through Alpaca minute bars for one time window."""
    rows = []
    page_token = None
    while True:
        params = {
            "symbols": ",".join(symbols),
            "timeframe": "1Min",
            "start": start.isoformat(),
            "end": end.isoformat(),
            "limit": 10000,
            "adjustment": "split",
            "feed": DATA_FEED,
            "sort": "asc",
        }
        if page_token:
            params["page_token"] = page_token

        response = None
        for attempt in range(5):
            response = session.get(ALPACA_BARS_URL, params=params, timeout=60)
            if response.status_code != 429:
                break
            retry_after = int(response.headers.get("Retry-After", "2"))
            time.sleep(retry_after + attempt)
        if response.status_code == 403:
            raise RuntimeError(
                "Alpaca rejected the data feed "
                f"'{DATA_FEED}'. The free plan uses iex. "
                "Set ALPACA_DATA_FEED=sip only if the account includes SIP. "
                f"Response: {response.text[:300]}"
            )
        response.raise_for_status()
        payload = response.json()
        for symbol, bars in (payload.get("bars") or {}).items():
            for bar in bars:
                ts = datetime.fromisoformat(bar["t"].replace("Z", "+00:00"))
                stored_symbol = symbol
                if symbol == "FB":
                    if ts >= META_RENAME:
                        continue
                    stored_symbol = "META"
                rows.append(
                    (
                        stored_symbol,
                        ts,
                        bar["o"],
                        bar["h"],
                        bar["l"],
                        bar["c"],
                        bar["v"],
                        bar.get("n"),
                        bar.get("vw"),
                        1 if symbol == "FB" else 0,
                    )
                )
        page_token = payload.get("next_page_token")
        if not page_token:
            return rows


def insert_rows(rows):
    if not rows:
        return 0
    # Prefer a native META bar over an FB bar mapped onto the same minute.
    deduped = {}
    rank = {}
    for row in rows:
        key = (row[0], row[1])
        row_rank = row[-1]
        if key not in deduped or row_rank < rank[key]:
            deduped[key] = row[:-1]
            rank[key] = row_rank
    values = list(deduped.values())
    with conn.transaction():
        with conn.cursor() as cur:
            cur.execute(
                """
                CREATE TEMP TABLE stock_minute_bars_stage (
                    symbol TEXT NOT NULL,
                    ts TIMESTAMPTZ NOT NULL,
                    open DOUBLE PRECISION NOT NULL,
                    high DOUBLE PRECISION NOT NULL,
                    low DOUBLE PRECISION NOT NULL,
                    close DOUBLE PRECISION NOT NULL,
                    volume BIGINT NOT NULL,
                    trade_count BIGINT,
                    vwap DOUBLE PRECISION
                ) ON COMMIT DROP
                """
            )
            with cur.copy(
                """
                COPY stock_minute_bars_stage (
                    symbol, ts, open, high, low, close, volume, trade_count, vwap
                ) FROM STDIN
                """
            ) as copy:
                for row in values:
                    copy.write_row(row)
            cur.execute(
                """
                INSERT INTO stock_minute_bars
                SELECT DISTINCT ON (symbol, ts)
                    symbol, ts, open, high, low, close, volume, trade_count, vwap
                FROM stock_minute_bars_stage
                ORDER BY symbol, ts
                ON CONFLICT (symbol, ts) DO NOTHING
                """
            )
            inserted = cur.rowcount
    return inserted


window_start = START
total_fetched = 0
total_inserted = 0
while window_start < END:
    window_end = min(window_start + WINDOW, END)
    symbols = ["AAPL", "META"]
    if window_start < META_RENAME:
        symbols.append("FB")
    rows = fetch_bars(symbols, window_start, window_end)
    inserted = insert_rows(rows)
    total_fetched += len(rows)
    total_inserted += inserted
    print(
        f"{window_start.date()} -> {window_end.date()}: "
        f"fetched {len(rows)} inserted {inserted}"
    )
    window_start = window_end

print(f"Done. fetched={total_fetched} inserted={total_inserted}")

In [ ]:
with conn.cursor(row_factory=dict_row) as cur:
    cur.execute(
        """
        SELECT symbol,
               count(*) AS rows,
               min(ts) AS first_bar,
               max(ts) AS last_bar
        FROM stock_minute_bars
        WHERE symbol = ANY(%s)
        GROUP BY symbol
        ORDER BY symbol
        """,
        (list(SYMBOLS),),
    )
    summary = cur.fetchall()

for row in summary:
    print(
        f"{row['symbol']}: rows={row['rows']} "
        f"first={row['first_bar']} last={row['last_bar']}"
    )

found = {row["symbol"] for row in summary}
missing_symbols = [symbol for symbol in SYMBOLS if symbol not in found]
if missing_symbols:
    raise RuntimeError(f"No bars stored for {', '.join(missing_symbols)}")

conn.close()